In [ ]:
# --- setup / imports (deduped) ---
import os 
os.chdir(r'/Users/sachuriga/Desktop/code/nwb4fp/SRC')

from neurochat.nc_data import NData
from neurochat.nc_spike import NSpike
from neurochat.nc_spatial import NSpatial
import neurochat.nc_plot as nc_plot
from neurochat.nc_lfp import NLfp
import matplotlib.pyplot as plt
import numpy as np
from pynwb import NWBHDF5IO
import math
import pynapple as nap
from scipy import signal
from sklearn.preprocessing import normalize

import sys
import nwb4fp.analyses.maps as mapp
from nwb4fp.analyses.examples.tracking_plot import plot_ratemap,plot_path
from nwb4fp.analyses.fields import separate_fields_by_laplace, separate_fields_by_dilation,find_peaks,separate_fields_by_laplace_of_gaussian,calculate_field_centers,distance_to_edge_function, remove_fields_by_area, map_pass_to_unit_circle,which_field,compute_crossings
from elephant.statistics import time_histogram, instantaneous_rate
from nwb4fp.analyses import maps
from nwb4fp.analyses.data import pos2speed,speed_filtered_spikes,load_speed_fromNWB,load_units_fromNWB,get_filed_num,unit_location_ch
from scipy.ndimage import gaussian_filter
import ast
import pandas as pd
pd.set_option('display.max_rows', None)
np.set_printoptions(threshold=np.inf)
import seaborn as sns
from scipy import stats
from scipy.ndimage import gaussian_filter1d
from scipy.stats import pearsonr
import matplotlib.gridspec as gridspec

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from mpl_toolkits.mplot3d import Axes3D
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings as _w


def _lmm_p(df, metric, control_ids, exp_ids):
    """Animal-level p: metric ~ genotype + (1|animal). Returns (p, n_cells, n_mice)."""
    d = df[['animal_id', metric]].copy()
    d['y'] = pd.to_numeric(d[metric], errors='coerce')
    d = d.dropna(subset=['y'])
    d['animal_id'] = d['animal_id'].astype(str)
    d['g'] = np.where(d['animal_id'].isin([str(x) for x in control_ids]), 'control', 'exp')
    d['g'] = pd.Categorical(d['g'], categories=['control', 'exp'])
    with _w.catch_warnings():
        _w.simplefilter('ignore')
        try:
            m = smf.mixedlm("y ~ g", d, groups=d['animal_id']).fit(reml=True)
            return m.pvalues.get('g[T.exp]', np.nan), len(d), d['animal_id'].nunique()
        except Exception:
            return np.nan, len(d), d['animal_id'].nunique()


def _celltype_gee_p(df, control_ids, exp_ids):
    """Animal-clustered binomial GEE for P(interneuron) ~ genotype. Returns (p, OR)."""
    d = df[df['buzaki_py_cell_type'].isin(['pyramidal', 'narrow_spike_interneurons'])].copy()
    d['animal_id'] = d['animal_id'].astype(str)
    d['is_int'] = (d['buzaki_py_cell_type'] == 'narrow_spike_interneurons').astype(float)
    d['g'] = np.where(d['animal_id'].isin([str(x) for x in control_ids]), 'control', 'exp')
    d['g'] = pd.Categorical(d['g'], categories=['control', 'exp'])
    with _w.catch_warnings():
        _w.simplefilter('ignore')
        try:
            m = smf.gee("is_int ~ g", "animal_id", d, family=sm.families.Binomial(),
                        cov_struct=sm.cov_struct.Exchangeable()).fit()
            return m.pvalues.get('g[T.exp]', np.nan), np.exp(m.params.get('g[T.exp]', np.nan))
        except Exception:
            return np.nan, np.nan

# --- 确保导出为 PDF 时文字在 Illustrator 中可编辑 ---
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
# -----------------------------------------------------

# 1. Load and Filter Data
# load the table, keep good units from session A
df_loaded = pd.read_pickle(r'/Users/sachuriga/Desktop/Projects/CR_CA1_paper/tables/functional_properties_with_python_measurements_with_stability.pkl')
df_good = df_loaded[df_loaded['unit_quality'] == "good"]
df_a = df_good[df_good['session'] == "A"]

# 2. Define Groups and Constants
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']
control_color = 'blue'
exp_color = "red"

# Separate into control and experimental groups
control_df = df_a[df_a['animal_id'].isin(control_ids)]
exp_df = df_a[df_a['animal_id'].isin(exp_ids)]

metrics = ['l_ratio', 'isi_violations_ratio', 'amplitude_median', 'snr']
titles = ['L ratio', 'ISI violations ratio', 'Amplitude median (µV)', 'Signal to noise ratio']
test_hy = ['two-sided', 'two-sided', 'two-sided', 'two-sided']

# 3. Setup Figure (保留了 dpi=1200)
fig = plt.figure(figsize=(7.2, 7.2), dpi=1200) 
plt.rcParams.update({'font.size': 7,'font.family': 'sans-serif','font.sans-serif':['Arial', 'Calibri','DejaVu Sans', 'sans-serif']})
plt.rcParams.update({
    'axes.labelpad': 5,
    'ytick.major.pad': 2,
    'xtick.major.pad': 5,
    'ytick.major.size': 2,
    'xtick.major.size': 2
})

gs = gridspec.GridSpec(3, 4, height_ratios=[.8, .8,.8], width_ratios=[0.8, .8, .8, .8])
ax1_1 = fig.add_subplot(gs[0, 0])
ax1_2 = fig.add_subplot(gs[0, 1])
ax1_3 = fig.add_subplot(gs[0, 2])
ax1_4 = fig.add_subplot(gs[0, 3])

axes_list = [ax1_1, ax1_2, ax1_3, ax1_4]
legend_stats = []
# # --- 4. Plotting Loop for Unit Quality Metrics ---
# one violin+box+strip panel per waveform / cell-type feature
for idx, ax in enumerate(axes_list):
    metric = metrics[idx]
    control_values = control_df[metric].dropna()
    exp_values = exp_df[metric].dropna()
    
    if len(control_values) > 0 and len(exp_values) > 0:
        # Animal-level test: LMM with animal as random intercept (not a cell-level
        # Mann-Whitney), so this QC control uses the same statistical unit as the
        # rest of the revision.
        p_val, n_cells, n_mice = _lmm_p(df_a, metric, control_ids, exp_ids)
        stat_desc = (f"{titles[idx]}: linear mixed-effects model (metric ~ genotype, "
                     f"animal random intercept), p = {p_val:.4f}, "
                     f"n = {n_cells} units from {n_mice} mice.")
        legend_stats.append(stat_desc)

        # Prepare data for Seaborn plotting
        plot_df = pd.DataFrame({
            'value': pd.concat([control_values, exp_values]),
            'group': ['Control'] * len(control_values) + ['Experimental'] * len(exp_values)
        })
        
        # Filter out outliers
        all_values = plot_df['value']
        mean_val = all_values.mean()
        std_val = all_values.std()
        plot_df_filtered = plot_df[(plot_df['value'] >= mean_val - 3 * std_val) & 
                                   (plot_df['value'] <= mean_val + 3 * std_val)]
        
        # --- Plotting ---
        # 1. Violin Plot
        sns.violinplot(
            data=plot_df_filtered, x='group', y='value', ax=ax, inner=None,
            palette={"Control": control_color, "Experimental": exp_color}, 
            width=0.8, cut=0, edgecolor='white', alpha=0.3
        )
        
        # 2. Box Plot
        sns.boxplot(
            data=plot_df_filtered, 
            x='group', y='value',
            palette={"Control": "black", "Experimental": "black"},
            width=0.3, fill=False, showfliers=False, showmeans=False, 
            linewidth=1, ax=ax, zorder=10
        )

        # 3. Strip Plot 
        sns.stripplot(
            data=plot_df_filtered,
            x='group', y='value',
            hue='group',
            palette={"Control": control_color, "Experimental": exp_color},
            size=2.5, alpha=0.6, jitter=True, dodge=False,
            ax=ax, legend=False, zorder=0
        )

        # Formatting
        ax.set_ylabel(titles[idx])
        ax.set_xlabel('')
        ax.yaxis.grid(False)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_visible(True)
        ax.spines['left'].set_visible(True)
        ax.set_xticklabels(['CR;DTA-', 'CR;DTA+'], rotation=-30)
        
        # Significance bars
        y_max = ax.get_ylim()[1]
        bar_height = y_max * 0.1 
        x_positions = [0, 1] 
        
        if pd.notna(p_val) and (p_val < 0.05) and (p_val > 0.01):
            ax.plot([x_positions[0], x_positions[1]], [y_max + bar_height, y_max + bar_height], color='black', lw=1.5)
            ax.text(0.5, y_max + bar_height * 1.1, f'*', ha='center', va='bottom')
        elif pd.notna(p_val) and (p_val < 0.01) and (p_val > 0.001):
            ax.plot([x_positions[0], x_positions[1]], [y_max + bar_height, y_max + bar_height], color='black', lw=1.5)
            ax.text(0.5, y_max + bar_height * 1.1, f'**', ha='center', va='bottom')
        elif pd.notna(p_val) and p_val < 0.001:
            ax.plot([x_positions[0], x_positions[1]], [y_max + bar_height, y_max + bar_height], color='black', lw=1.5)
            ax.text(0.5, y_max + bar_height * 1.1, f'***', ha='center', va='bottom')

# 5. Data Prep for 3D Scatter & Bar Charts (Cell Types)
df_ko_py = df_good[(df_good['buzaki_py_cell_type']=='pyramidal') & (df_good['session'] == 'A') & (df_good['genotype']=='NDNF-flp-/- and Pde1c -/-')]
df_het_py = df_good[(df_good['buzaki_py_cell_type']=='pyramidal') & (df_good['session'] == 'A') & (df_good['genotype']=='NDNF-flp +/- and Pde1c +/-')]
df_ko_int = df_good[(df_good['buzaki_py_cell_type']=='narrow_spike_interneurons') & (df_good['session'] == 'A') & (df_good['genotype']=='NDNF-flp-/- and Pde1c -/-')]
df_het_int = df_good[(df_good['buzaki_py_cell_type']=='narrow_spike_interneurons') & (df_good['session'] == 'A') & (df_good['genotype']=='NDNF-flp +/- and Pde1c +/-')]

s=10
lw=.31
ax = fig.add_subplot(gs[1:2, 0:3], projection='3d')

# Scatter plots
ax.scatter(df_ko_py['Averate_rate'], df_ko_py['peak_to_valley']*1000, df_ko_py['mean_inter_spike_interval'], c='blue', marker='^', s=s, edgecolors='white', linewidths=lw, alpha=1)
ax.scatter(df_ko_int['Averate_rate'], df_ko_int['peak_to_valley']*1000, df_ko_int['mean_inter_spike_interval'], c='cyan', marker='o', s=s, edgecolors='white', linewidths=lw, alpha=1)
ax.scatter(df_het_py['Averate_rate'], df_het_py['peak_to_valley']*1000, df_het_py['mean_inter_spike_interval'], c='red', marker='^', s=s, edgecolors='white', linewidths=lw, alpha=1)
ax.scatter(df_het_int['Averate_rate'], df_het_int['peak_to_valley']*1000, df_het_int['mean_inter_spike_interval'], c='magenta', marker='o', s=s, edgecolors='white', linewidths=lw, alpha=1)

ax.set_ylim(0,1.25)
ax.set_xlabel('Firing rate(Hz)')
ax.set_ylabel('Peak to valley (ms)')
ax.set_zlabel('Mean inter spike interval (ms)')

# legend: marker shape = cell type, colour = genotype
from matplotlib.lines import Line2D
_leg = [
    Line2D([0],[0], marker='^', color='none', markerfacecolor='blue',    markeredgecolor='white', markersize=6, label='Pyr. CR;DTA-'),
    Line2D([0],[0], marker='^', color='none', markerfacecolor='red',     markeredgecolor='white', markersize=6, label='Pyr. CR;DTA+'),
    Line2D([0],[0], marker='o', color='none', markerfacecolor='cyan',    markeredgecolor='white', markersize=6, label='Int. CR;DTA-'),
    Line2D([0],[0], marker='o', color='none', markerfacecolor='magenta', markeredgecolor='white', markersize=6, label='Int. CR;DTA+'),
]
ax.legend(handles=_leg, fontsize=5.5, frameon=False, loc='upper left',
          bbox_to_anchor=(-0.02, 1.02), handletextpad=0.3, labelspacing=0.3)

# 6. Bar Chart calculations
con_all = len(df_ko_py) + len(df_ko_int)
exp_all = len(df_het_py) + len(df_het_int)

groups = ['CR;DTA-', 'CR;DTA+']
counts = [
    [round(len(df_ko_py)/con_all,2)*100, round(len(df_ko_int)/con_all,2)*100], 
    [round(len(df_het_py)/exp_all,2)*100, round(len(df_het_int)/exp_all,2)*100], 
]
percentages = counts 
con_py, con_int = len(df_ko_py), len(df_ko_int)
exp_py, exp_int = len(df_het_py), len(df_het_int)

# Cell-type proportion: animal-clustered binomial GEE (not a cell-level chi-square)
p_ct, or_ct = _celltype_gee_p(df_a, control_ids, exp_ids)
legend_stats.append(f"Cell-type proportion (interneuron vs pyramidal): binomial GEE "
                    f"clustered on animal, p = {p_ct:.4f}, OR = {or_ct:.2f}, "
                    f"total n = {con_py + con_int + exp_py + exp_int} units from "
                    f"{df_a['animal_id'].astype(str).nunique()} mice.")
ax1 = fig.add_subplot(gs[1:2, 3])
colors = ['blue', "cyan"]  
colors1 = ['red', "magenta"]    

# Adjust bar chart height
pos = ax1.get_position()
ax1.set_position([pos.x0, pos.y0 + 0.2, pos.width, pos.height * 0.5])

bar_width = 0.35 
x = np.arange(len(groups)) * 0.5

bars = []
# stacked bars for the cell-type proportions
for i in range(len(counts[0])): 
    bar = ax1.bar(x, [counts[j][i] for j in range(len(counts))], 
                 bottom=[sum(counts[j][:i]) for j in range(len(counts))], 
                 color=[colors[i] if j == 0 else colors1[i] for j in range(len(counts))], 
                 label=f'Category {i+1}' if i == 0 else None, 
                 width=bar_width)
    bars.append(bar)

for i, bar_group in enumerate(bars):
    cell_type = "(Pyramidal)" if i == 0 else "(Interneurons)"
    for j, bar in enumerate(bar_group):
        height = bar.get_height()
        bottom = sum(counts[j][:i])
        ax1.text(bar.get_x() + bar.get_width()/2, bottom + height/2, 
                f'{percentages[j][i]}%\n {cell_type}', ha='center', va='center', rotation=0, color='black',fontsize=7)

ax1.set_ylabel('%Neurons')
ax1.set_xticks(x)
ax1.set_xticklabels(groups, rotation = -45)
ax1.legend().set_visible(False)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# 7. Save Figure (保留了 dpi=1200, 格式为 PDF)
fig.subplots_adjust(top=0.92, bottom=0.08, left=0.1, right=0.95, hspace=1.1, wspace=1.3)
plt.tight_layout()
plt.savefig(r'/Users/sachuriga/Desktop/Projects/CR_CA1_paper/Figures_neuron_report_raw/suppfig4.pdf', transparent=True, dpi=1200, bbox_inches='tight')

In [ ]:
legend_stats